# 02. Stage 1: Domain-Adaptive Pretraining (DAPT)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)
### Academic Research Project: **GeoFinBERT**

---
### Architectural Methodology:
1. **Base Model**: Load `ProsusAI/FinBERT`, which is fine-tuned on corporate financial filings.
2. **Architecture Transfer**: Transfer FinBERT's 12-layer BERT transformer encoder and embeddings into a `BertForMaskedLM` architecture.
3. **MLM Objective**: Masked Language Modeling dynamically masks 15% of tokens (80% `[MASK]`, 10% random, 10% unchanged) over authentic geopolitical & energy market news.
4. **Tokenizer Preservation**: Retain FinBERT's vocabulary of 30,522 tokens and tie prediction embeddings.
5. **Checkpointing**: Save the domain-adapted checkpoint as `GeoFinBERT-DAPT` alongside training logs, validation loss, and perplexity.


In [ ]:
# Setup and Hardware Diagnostics
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("[Colab Environment Detected] Installing dependencies...")
    !pip install -q -r requirements.txt
    !pip install -q -e .

import torch
import pandas as pd
from torch.utils.data import DataLoader

sys.path.append(os.path.abspath("src"))

from geofinbert.config import load_config
from geofinbert.utils import set_seed, get_device_info, count_parameters
from geofinbert.models.dapt import build_dapt_model
from geofinbert.data.dataset import MLMTextDataset, get_mlm_collator
from geofinbert.training.trainer_dapt import run_dapt_training

config = load_config("configs/config.yaml")
set_seed(config["project"]["seed"])

device_info = get_device_info()
print("Hardware Check:", device_info)
if device_info["cuda_available"]:
    print(f"Accelerated GPU Active: {device_info['gpu_name']} ({device_info['vram_gb']} GB VRAM)")
else:
    print("Warning: CUDA GPU not detected. For fast GPU training in Colab: Runtime -> Change runtime type -> T4/A100 GPU.")


## Step 1: Load Chronological DAPT Partitions


In [ ]:
splits_dir = config["data"]["splits_dir"]
train_df = pd.read_parquet(os.path.join(splits_dir, "dapt_train.parquet"))
val_df = pd.read_parquet(os.path.join(splits_dir, "dapt_val.parquet"))

print(f"Loaded DAPT Training Set: {len(train_df):,} articles")
print(f"Loaded DAPT Validation Set: {len(val_df):,} articles")


## Step 2: Build Architecture & Transfer FinBERT Weights
We instantiate `BertForMaskedLM` using `ProsusAI/FinBERT` encoder weights, keeping the original tokenizer intact.


In [ ]:
base_model_name = config["model"]["base_model_name"]
model, tokenizer = build_dapt_model(
    base_model_name_or_path=base_model_name,
    tokenizer_name=config["model"]["tokenizer_name"],
    local_files_only=False
)

total_p, train_p = count_parameters(model)
print(f"Base FinBERT Encoder successfully transferred to BertForMaskedLM.")
print(f"Total Parameters: {total_p:,} | Trainable: {train_p:,}")
print(f"Vocabulary Size: {tokenizer.vocab_size}")


## Step 3: Prepare Dynamic MLM DataLoaders
Hugging Face's dynamic masking collator (`DataCollatorForLanguageModeling`) masks 15% of tokens dynamically each epoch.


In [ ]:
max_seq_len = config["model"].get("max_seq_length", 128)
train_dataset = MLMTextDataset(train_df["text"].tolist(), tokenizer, max_length=max_seq_len)
val_dataset = MLMTextDataset(val_df["text"].tolist(), tokenizer, max_length=max_seq_len)

collator = get_mlm_collator(tokenizer, mlm_probability=config["stage1_dapt"].get("mlm_probability", 0.15))

batch_size = config["stage1_dapt"].get("per_device_train_batch_size", 16)
eval_batch_size = config["stage1_dapt"].get("per_device_eval_batch_size", 32)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collator, num_workers=2 if device_info["cuda_available"] else 0)
val_loader = DataLoader(val_dataset, batch_size=eval_batch_size, shuffle=False, collate_fn=collator, num_workers=2 if device_info["cuda_available"] else 0)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


## Step 4: Execute Domain-Adaptive Pretraining
We optimize with AdamW, linear learning-rate warmup, gradient clipping, and mixed-precision (AMP) for GPU acceleration.


In [ ]:
output_dir = config["stage1_dapt"]["output_dir"]
dapt_cfg = config["stage1_dapt"]

dapt_log = run_dapt_training(
    model=model,
    tokenizer=tokenizer,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    output_dir=output_dir,
    num_epochs=dapt_cfg["num_train_epochs"],
    learning_rate=float(dapt_cfg["learning_rate"]),
    weight_decay=float(dapt_cfg["weight_decay"]),
    warmup_ratio=float(dapt_cfg["warmup_ratio"]),
    use_fp16=dapt_cfg["fp16"] and device_info["cuda_available"]
)


## Step 5: Visualize DAPT Pretraining Convergence
We inspect training loss and validation perplexity across epochs.


In [ ]:
import matplotlib.pyplot as plt

history = dapt_log["history"]
epochs = [h["epoch"] for h in history]
train_losses = [h["train_loss"] for h in history]
val_losses = [h["val_loss"] for h in history]
perplexities = [h["perplexity"] for h in history]

fig, ax1 = plt.subplots(figsize=(8, 4))

ax1.plot(epochs, train_losses, "b-o", label="Train Loss (MLM)")
ax1.plot(epochs, val_losses, "r-s", label="Val Loss (MLM)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross Entropy Loss")
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(epochs, perplexities, "g--^", label="Validation Perplexity")
ax2.set_ylabel("Perplexity (exp(loss))", color="g")
ax2.tick_params(axis="y", labelcolor="g")

plt.title("GeoFinBERT-DAPT Pretraining Convergence")
plt.tight_layout()
plt.show()


## Step 6: Verify Checkpoint Compatibility
We verify that the saved checkpoint `GeoFinBERT-DAPT` can be loaded seamlessly for Stage 2 fine-tuning.


In [ ]:
from transformers import BertForMaskedLM
loaded_dapt = BertForMaskedLM.from_pretrained(output_dir)
print(f"Verified GeoFinBERT-DAPT checkpoint at {output_dir}")
print(f"Checkpoint parameter count: {sum(p.numel() for p in loaded_dapt.parameters()):,}")
